# RAG Application

![Simple RAG](../../images/simple_rag.png)

In this notebook, we're going to set up a simple RAG application that we'll be using as we learn more about LangSmith.

RAG (Retrieval Augmented Generation) is a popular technique for providing LLMs with relevant documents that will enable them to better answer questions from users. 

In our case, we are going to index some LangSmith documentation!

LangSmith makes it easy to trace any LLM application, no LangChain required!

### Setup

Make sure you set your environment variables, including your OpenAI API key.

In [ ]:
# You can set them inline!
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### Simple RAG application

In [2]:
from langsmith import traceable
from openai import OpenAI
from typing import List, Dict, Any
import nest_asyncio
import requests
import json
from urllib.parse import urljoin, quote
import time
from dataclasses import dataclass
from bs4 import BeautifulSoup
import re

MODEL_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context from DevDocs.io to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
Include relevant code examples when appropriate.
"""

@dataclass
class Document:
    page_content: str
    metadata: Dict[str, Any]

class DevDocsRetriever:
    """Retriever for DevDocs.io documentation"""
    
    def __init__(self, docs_list: List[str] = None):
        """
        Initialize with list of documentation sources
        Example docs: ['javascript', 'python~3.11', 'react', 'node', 'css']
        """
        self.base_url = "https://devdocs.io"
        self.docs_list = docs_list or ['javascript', 'python~3.11', 'react', 'css']
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (compatible; RAG-Bot/1.0)'
        })
    
    def search_devdocs(self, query: str, doc_type: str = None, limit: int = 5) -> List[Dict]:
        """Search DevDocs.io for relevant entries"""
        results = []
        
        docs_to_search = [doc_type] if doc_type else self.docs_list
        
        for doc in docs_to_search:
            try:
                # Get the index for the documentation
                index_url = f"{self.base_url}/docs/{doc}/index.json"
                response = self.session.get(index_url, timeout=10)
                
                if response.status_code == 200:
                    index_data = response.json()
                    
                    # Search through entries
                    query_lower = query.lower()
                    matching_entries = []
                    
                    for entry in index_data.get('entries', []):
                        name = entry.get('name', '').lower()
                        path = entry.get('path', '').lower()
                        
                        # Score entries based on relevance
                        score = 0
                        if query_lower in name:
                            score += 10
                        if any(term in name for term in query_lower.split()):
                            score += 5
                        if query_lower in path:
                            score += 3
                        
                        if score > 0:
                            matching_entries.append({
                                'entry': entry,
                                'doc_type': doc,
                                'score': score
                            })
                    
                    # Sort by score and take top results
                    matching_entries.sort(key=lambda x: x['score'], reverse=True)
                    results.extend(matching_entries[:limit])
                    
            except Exception as e:
                print(f"Error searching {doc}: {e}")
                continue
            
            # Rate limiting
            time.sleep(0.1)
        
        # Sort all results by score and return top entries
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:limit * 2]  # Return more results for better context
    
    def fetch_content(self, doc_type: str, path: str) -> str:
        """Fetch and extract clean content from DevDocs.io using Beautiful Soup"""
        try:
            content_url = f"{self.base_url}/docs/{doc_type}/{path}"
            response = self.session.get(content_url, timeout=10)
            
            if response.status_code == 200:
                return self.extract_clean_content(response.text, content_url)
            else:
                return f"Could not fetch content from {content_url} (Status: {response.status_code})"
                
        except Exception as e:
            return f"Error fetching content: {e}"
    
    def extract_clean_content(self, html_content: str, url: str) -> str:
        """Extract clean, readable content from HTML using Beautiful Soup"""
        try:
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # DevDocs.io typically has content in specific containers
            # Try to find the main content area
            content_selectors = [
                '._content',
                '.content',
                '#content', 
                'main',
                '.documentation',
                'article',
                '.entry-content'
            ]
            
            main_content = None
            for selector in content_selectors:
                main_content = soup.select_one(selector)
                if main_content:
                    break
            
            if not main_content:
                # Fallback: try to find content by removing common non-content elements
                for unwanted in soup(['script', 'style', 'nav', 'header', 'footer', 'aside']):
                    unwanted.decompose()
                main_content = soup.find('body') or soup
            
            # Clean up the content
            if main_content:
                # Remove unwanted elements
                for unwanted in main_content(['script', 'style', 'nav', 'header', 'footer', 
                                            'aside', '.sidebar', '.navigation', '.ads']):
                    unwanted.decompose()
                
                # Extract text while preserving some structure
                text_content = self.html_to_structured_text(main_content)
                
                # Clean up whitespace
                text_content = re.sub(r'\n\s*\n', '\n\n', text_content)
                text_content = re.sub(r' +', ' ', text_content)
                text_content = text_content.strip()
                
                return text_content[:3000]  # Limit content size
            else:
                return f"Could not extract content from {url}"
                
        except Exception as e:
            return f"Error parsing HTML content: {e}"
    
    def html_to_structured_text(self, element) -> str:
        """Convert HTML element to structured text while preserving important formatting"""
        text_parts = []
        
        for child in element.descendants:
            if child.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
                text_parts.append(f"\n\n## {child.get_text(strip=True)}\n")
            elif child.name == 'p':
                text = child.get_text(strip=True)
                if text:
                    text_parts.append(f"{text}\n\n")
            elif child.name in ['code', 'tt']:
                text = child.get_text(strip=True)
                if text and len(text) > 1:  # Avoid single character code spans
                    text_parts.append(f"`{text}`")
            elif child.name == 'pre':
                code_text = child.get_text(strip=True)
                if code_text:
                    text_parts.append(f"\n```\n{code_text}\n```\n")
            elif child.name in ['ul', 'ol']:
                # Handle lists
                for li in child.find_all('li', recursive=False):
                    li_text = li.get_text(strip=True)
                    if li_text:
                        text_parts.append(f"• {li_text}\n")
                text_parts.append("\n")
            elif child.name == 'strong' or child.name == 'b':
                text = child.get_text(strip=True)
                if text:
                    text_parts.append(f"**{text}**")
            elif child.name == 'em' or child.name == 'i':
                text = child.get_text(strip=True)
                if text:
                    text_parts.append(f"*{text}*")
            elif child.string and child.parent.name not in ['script', 'style']:
                # Regular text node
                text = child.string.strip()
                if text and not any(tag in str(child.parent) for tag in ['<code', '<pre', '<h1', '<h2', '<h3', '<h4', '<h5', '<h6', '<p']):
                    text_parts.append(text + " ")
        
        return ''.join(text_parts)
    
    def invoke(self, question: str) -> List[Document]:
        """Main retrieval method compatible with the original interface"""
        # Search for relevant entries
        search_results = self.search_devdocs(question)
        
        documents = []
        for result in search_results:
            entry = result['entry']
            doc_type = result['doc_type']
            
            # Fetch the actual content
            content = self.fetch_content(doc_type, entry.get('path', ''))
            
            # Create document object
            doc = Document(
                page_content=f"# {entry.get('name', 'Unknown')} ({doc_type})\n"
                           f"Source: devdocs.io/{doc_type}/{entry.get('path', '')}\n\n"
                           f"{content}",
                metadata={
                    'source': f"devdocs.io/{doc_type}",
                    'title': entry.get('name', ''),
                    'path': entry.get('path', ''),
                    'doc_type': doc_type,
                    'score': result['score'],
                    'url': f"https://devdocs.io/{doc_type}#{entry.get('path', '')}"
                }
            )
            documents.append(doc)
        
        return documents

openai_client = OpenAI()
nest_asyncio.apply()

# Initialize DevDocs retriever with common documentation sources
retriever = DevDocsRetriever([
    'javascript',
    'python~3.11', 
    'react',
    'node',
    'css',
    'html',
    'typescript',
    'express',
    'django~4.2'
])

@traceable(run_type="chain")
def retrieve_documents(question: str):
    """
    retrieve_documents
    - Returns documents fetched from DevDocs.io based on the user's question
    """
    return retriever.invoke(question)

@traceable(run_type="chain")
def generate_response(question: str, documents):
    """
    generate_response
    - Calls `call_openai` to generate a model response after formatting inputs
    """
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

@traceable(run_type="llm")
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    """
    call_openai
    - Returns the chat completion output from OpenAI
    """
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

@traceable(run_type="chain")
def langsmith_rag(question: str):
    """
    langsmith_rag
    - Calls `retrieve_documents` to fetch documents from DevDocs.io
    - Calls `generate_response` to generate a response based on the fetched documents
    - Returns the model response
    """
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

# Additional utility functions for specific documentation types
def search_specific_docs(question: str, doc_types: List[str]):
    """Search only specific documentation types"""
    specific_retriever = DevDocsRetriever(doc_types)
    return specific_retriever.invoke(question)

# Example usage:
if __name__ == "__main__":
    # Test the RAG system
    test_questions = [
        "How do I use fetch in JavaScript?",
        "What is useState in React?",
        "How to create a virtual environment in Python?",
        "CSS flexbox properties"
    ]
    
    for question in test_questions:
        print(f"Question: {question}")
        try:
            answer = langsmith_rag(question)
            print(f"Answer: {answer}\n")
        except Exception as e:
            print(f"Error: {e}\n")

Question: How do I use fetch in JavaScript?
Answer: To use the `fetch` API in JavaScript, you can call it with a URL and it returns a promise that resolves to the response. Here's a basic example:

```javascript
fetch('https://api.example.com/data')
  .then(response => response.json())
  .then(data => console.log(data))
  .catch(error => console.error('Error:', error));
```

This code fetches data from the specified URL, converts the response to JSON, and logs it to the console.

Question: What is useState in React?
Answer: The `useState` hook in React is a function that allows you to add state to functional components. It returns an array with two elements: the current state value and a function to update that state. Here's a simple example:

```javascript
import React, { useState } from 'react';

function Counter() {
  const [count, setCount] = useState(0);

  return (
    <div>
      <p>You clicked {count} times</p>
      <button onClick={() => setCount(count + 1)}>Click me</button>

This should take a little less than a minute. We are indexing and storing LangSmith documentation in a SKLearn vector database.

In [3]:
question = "What is LangSmith used for?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"website": "www.google.com"}})
print(ai_answer)

I don't know.


### Let's take a look in LangSmith!